Layer Normalization

In [13]:
import torch 
import torch.nn as nn 
class LayerNormFromScratch(nn.Module):
    def __init__(self, emb_dim, eps=1e-5):
        super().__init__()
        self.eps=eps
        self.scale=nn.Parameter(torch.ones(emb_dim))
        self.shift=nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean= x.mean(dim=-1, keepdim=True)
        var= x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x=(x-mean)/torch.sqrt(var+self.eps)
        return self.scale*norm_x+self.shift

Feed Tensors into LayerNorm

In [14]:
dummy_inputs= torch.randn(2,3,5)
ln_layer= LayerNormFromScratch(emb_dim=5)
ln_outputs= ln_layer(dummy_inputs)

print("input tensor shape:", dummy_inputs.shape)
print("output tensor shape:", ln_outputs.shape)
print("\n normalised output mean (should be close to 0):\n", ln_outputs.mean(dim=-1, keepdim=True))
print("\n normalised output var(should be close to 1):\n", ln_outputs.var(dim=-1, keepdim=True, unbiased=False))

input tensor shape: torch.Size([2, 3, 5])
output tensor shape: torch.Size([2, 3, 5])

 normalised output mean (should be close to 0):
 tensor([[[ 3.5763e-08],
         [ 5.9605e-09],
         [ 1.1921e-08]],

        [[-8.3447e-08],
         [ 1.1921e-08],
         [-2.0862e-08]]], grad_fn=<MeanBackward1>)

 normalised output var(should be close to 1):
 tensor([[[1.0000],
         [1.0000],
         [1.0000]],

        [[1.0000],
         [1.0000],
         [1.0000]]], grad_fn=<VarBackward0>)


GELU Activation Function(Gaussian Linear Unit)

In [15]:
import torch 
import torch.nn as nn
class GELUActivation(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5*x*(1.0+torch.tanh(torch.sqrt(torch.tensor(2.0/torch.pi))*(x+0.044715*torch.pow(x,3))))

Implementing the Feed-Forward Network Block

In [16]:
import torch
import torch.nn as nn
class FeedForward(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.layers=nn.Sequential(nn.Linear(emb_dim, 4*emb_dim), GELUActivation(), nn.Linear(4*emb_dim, emb_dim))

    def forward(self,x):
        return self.layers(x)

The dimensional Verification Test 


In [17]:
batch_inputs= torch.randn(8,4,256)
ffn_block=FeedForward(emb_dim=256)
ffn_outputs= ffn_block(batch_inputs)
print("input batch shape:", batch_inputs.shape)
print("ffn output shape:", ffn_outputs.shape)

input batch shape: torch.Size([8, 4, 256])
ffn output shape: torch.Size([8, 4, 256])


Residual Connections 

In [18]:
%pip install import-ipynb

Note: you may need to restart the kernel to use updated packages.


In [19]:
%run ../02_attention/attention.ipynb

inputs shape: torch.Size([3, 3])
query: tensor([0.4400, 0.1500, 0.8900])
raw attention scores: tensor([1.0082, 0.9599, 0.4495])
attention weights(percentages): tensor([0.3961, 0.3774, 0.2265])
final context vector: tensor([0.5070, 0.5803, 0.6197])
all raw scores matrix:
 tensor([[0.9995, 0.9544, 0.4438],
        [0.9544, 1.4950, 1.1058],
        [0.4438, 1.1058, 1.0538]])

all attention weights matrix(row-wise sum=1):
 tensor([[0.3953, 0.3779, 0.2268],
        [0.2577, 0.4425, 0.2998],
        [0.2092, 0.4057, 0.3851]])

all final context vectors matrix:
 tensor([[0.5071, 0.5808, 0.6194],
        [0.5251, 0.6785, 0.5454],
        [0.5326, 0.7116, 0.4848]])
queries matrix shape: torch.Size([3, 2])
keys matrix shape: torch.Size([3, 2])
values matrix shape: torch.Size([3, 2])
scaled attention weights matrix:
 tensor([[0.3710, 0.3940, 0.2350],
        [0.3888, 0.4236, 0.1877],
        [0.3765, 0.3983, 0.2252]])

 final trainable context vector matrix shape: torch.Size([3, 2])
causal (GPT-s

In [20]:
import torch
import torch.nn as nn 
class TransformerBlock(nn.Module):
    def __init__(self, emb_dim, context_length, dropout, num_heads):
        super().__init__()
        self.att=MultiHeadAttentionWrapper(d_in=emb_dim, d_out=emb_dim, context_length=context_length, dropout=dropout, num_heads=num_heads)
        self.ff= FeedForward(emb_dim=emb_dim)
        self.ln1= LayerNormFromScratch(emb_dim=emb_dim)
        self.ln2= LayerNormFromScratch(emb_dim=emb_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self,x):
        x=x+self.dropout(self.att(self.ln1(x)))
        x=x+self.dropout(self.ff(self.ln2(x)))

        return x 

Transformer Test 

In [21]:
batch_inputs=torch.randn(8,4,256)
gpt_block= TransformerBlock(emb_dim=256, context_length=4, dropout=0.1, num_heads=4)
block_outputs= gpt_block(batch_inputs)
print("input batch shape:", batch_inputs.shape)
print("transformer block out shape:", block_outputs.shape)


input batch shape: torch.Size([8, 4, 256])
transformer block out shape: torch.Size([8, 4, 256])


In [22]:
class GPTMODEL(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.tok_emb=nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb= nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb= nn.Dropout(cfg["drop_rate"])

        self.trf_blocks=nn.Sequential(*[TransformerBlock(emb_dim=cfg["emb_dim"], context_length=cfg["context_length"], 
        dropout=cfg["drop_rate"], num_heads=cfg["n_heads"])
        for _ in range(cfg["n_layers"])])
        self.final_ln= LayerNormFromScratch(emb_dim=cfg["emb_dim"])
        self.out_head= nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self,in_idx):
        b,seq_len=in_idx.shape
        tok_embeds= self.tok_emb(in_idx)
        pos_embeds= self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x= self.drop_emb(tok_embeds+pos_embeds)
        x= self.trf_blocks(x)
        x=self.final_ln(x)
        logits= self.out_head(x)
        return logits 

In [23]:
GPT_CONFIG_124M= {"vocab_size": 50257, "context_length": 256, "emb_dim": 768, "n_heads": 12, "n_layers": 12, "drop_rate": 0.1, "qkv_bias": False}
torch.manual_seed(123)
model=GPTMODEL(GPT_CONFIG_124M)
sample_token_ids=torch.tensor([[505, 707, 808]])
output_logits = model(sample_token_ids)
print("input token batch shape:", sample_token_ids.shape)
print("final output logits shape:", output_logits.shape)

input token batch shape: torch.Size([1, 3])
final output logits shape: torch.Size([1, 3, 50257])


Parameter Count & The Generation Loop

In [26]:
total_params= sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"total trainable parameter: {total_params:,}")

def generate_text_simple(model, idx, max_new_tokens, context_size):
    for _ in range(max_new_tokens):
        idx_cond= idx[:, -context_size:]

        with torch.no_grad():
            logits= model(idx_cond)
        logits = logits[:, -1, :]
        next_token= torch.argmax(logits, dim=-1, keepdim=True)
        idx=torch.cat((idx, next_token), dim=-1)
    return idx
start_context=torch.tensor([[40,367,2885]])

encoded_output= generate_text_simple(model, idx=start_context, max_new_tokens=5, context_size=GPT_CONFIG_124M["context_length"])

print("\nfinal encoded token ids output:", encoded_output)

total trainable parameter: 162,419,712

final encoded token ids output: tensor([[   40,   367,  2885, 31748, 14767, 40404, 24427, 44724]])
